# Fin - Educador Financeiro Inteligente

Desafio Final Bootcamp DIO - Assistente Virtual com IA.

Arquitetura RAG: recuperação de dados (perfil, transações, produtos) + geração via LLM (Gemini) com system prompt anti-alucinação, e fallback determinístico por regras caso a IA não esteja disponível.

Execute as células em ordem. A Célula 3 pede uma API key gratuita do Google AI Studio (https://aistudio.google.com/apikey) — se você apertar Enter sem colar nada, o assistente roda só no modo por regras.

In [ ]:
# !pip install -q pandas google-genai

In [1]:
import re
import pandas as pd

perfil = {
    "nome": "Lucas",
    "cidade": "Belo Horizonte, MG",
    "renda_mensal": 8500,
    "perfil_risco": "moderado",
    "score_risco": 65,
    "objetivos": [{"nome": "Reserva de emergência", "meta": 10000, "atual": 3200}],
}

produtos_data = {
    "produtos": [
        {"id": "tesouro_selic", "nome": "Tesouro Selic", "risco": "baixo",
         "rentabilidade": "100% Selic", "liquidez": "D+1", "minimo": 30},
        {"id": "cdb_110", "nome": "CDB 110% CDI", "risco": "baixo",
         "rentabilidade": "110% CDI", "liquidez": "Diária", "minimo": 100},
        {"id": "etf_ivvb11", "nome": "ETF IVVB11 (S&P500)", "risco": "alto",
         "rentabilidade": "variável", "liquidez": "D+2", "minimo": 300},
    ],
    "faq": [
        {"pergunta": "O que é CDI?",
         "resposta": "CDI é a taxa entre bancos, próxima da Selic, benchmark da renda fixa."}
    ],
}

transacoes = pd.DataFrame([
    {"data": "2026-01-05", "descricao": "Salário", "valor": 8500, "tipo": "entrada"},
    {"data": "2026-01-06", "descricao": "Aluguel", "valor": -1800, "tipo": "saida"},
    {"data": "2026-01-08", "descricao": "Mercado", "valor": -650, "tipo": "saida"},
    {"data": "2026-02-05", "descricao": "Salário", "valor": 8500, "tipo": "entrada"},
    {"data": "2026-03-18", "descricao": "Reserva", "valor": -500, "tipo": "saida"},
])


def calcular_resumo():
    # aportes de reserva não contam como despesa no custo médio
    gastos = transacoes[(transacoes["tipo"] == "saida") & (transacoes["descricao"] != "Reserva")]
    custo = gastos["valor"].abs().mean()
    meta = perfil["objetivos"][0]["meta"]
    atual = perfil["objetivos"][0]["atual"]
    falta = meta - atual
    return {
        "custo_medio": custo,
        "meta": meta,
        "atual": atual,
        "falta": falta,
        "progresso": atual / meta * 100,
    }


def responder_regras(pergunta: str) -> str:
    """Motor determinístico por palavras-chave. Usado como fallback quando a IA
    generativa não está disponível (sem chave, sem internet, erro de cota etc.)."""
    q = pergunta.lower().strip()
    r = calcular_resumo()

    if not q:
        return "Pode repetir sua pergunta? Não recebi nenhum texto."

    if "garantido" in q and "%" in q:
        return ("Não existe investimento com rentabilidade garantida desse tipo — isso é sinal de golpe. "
                "Na nossa base, a maior opção é CDB 110% CDI (~0,9% a.m.). Fonte: produtos_financeiros.json")

    if any(k in q for k in ["reserva", "falta", "meta"]):
        return (f"Reserva: meta R${r['meta']:.0f}, atual R${r['atual']:.0f} ({r['progresso']:.0f}%). "
                f"Falta R${r['falta']:.0f}. Fonte: perfil_investidor.json")

    if any(k in q for k in ["cdi", "selic", "juro", "juros", "rende", "rentabilidade"]):
        return ("CDI é a taxa entre bancos, próxima da Selic, benchmark da renda fixa. "
                "Para perfil moderado: Tesouro Selic (risco baixo, D+1, a partir de R$30) "
                "e CDB 110% CDI (risco baixo, liquidez diária). Fonte: produtos_financeiros.json")

    if any(k in q for k in ["viagem", "posso gastar", "gastar"]):
        m = re.search(r"(\d+)", q)
        valor = int(m.group(1)) if m else 900
        pct = valor / r["custo_medio"] * 100
        return (f"Seu custo médio mensal (sem contar aportes) é de R${r['custo_medio']:.0f}. "
                f"Gastar R${valor} representa {pct:.0f}% disso. Sua reserva está em {r['progresso']:.0f}% — "
                f"um gasto grande pode atrasar sua meta. Fonte: transacoes.csv + perfil_investidor.json")

    if any(k in q for k in ["ivvb11", "etf"]):
        return ("IVVB11 é um ETF que replica o S&P500 — risco alto, liquidez D+2, aplicação mínima R$300. "
                "Para perfil moderado, recomenda-se só uma parcela pequena (10-20%) do total investido, "
                "e de longo prazo. Para a reserva de emergência, prefira Tesouro Selic ou CDB. "
                "Fonte: produtos_financeiros.json")

    return ("Não tenho essa informação na minha base de conhecimento. "
            "Posso ajudar com: reserva de emergência, CDI/Selic, CDB, IVVB11 ou se um gasto cabe no seu orçamento. "
            "Pode reformular a pergunta usando um desses temas?")

In [2]:
import getpass

SYSTEM_PROMPT = """Você é Fin, um educador financeiro virtual, amigável e direto.
Responda SOMENTE com base no CONTEXTO fornecido abaixo — nunca invente números,
taxas, produtos ou promessas de rentabilidade que não estejam nele.
Se a resposta não estiver no CONTEXTO, diga claramente que não tem essa informação
e liste os temas que você sabe responder (reserva, CDI/Selic, CDB, IVVB11, orçamento).
Nunca confirme rentabilidade "garantida" fora do que está no CONTEXTO — alerte que é
sinal de golpe. Sempre cite a fonte usada (nome do arquivo) ao final da resposta.
Responda em no máximo 3 frases, em português, sem jargão sem explicar."""

_MODEL_DISPONIVEL = False
_client = None
_MODEL_NOME = "gemini-3.1-flash-lite"  # estável, gratuito no free tier

try:
    from google import genai

    # Chave gratuita em https://aistudio.google.com/apikey — não fica salva no notebook.
    _api_key = getpass.getpass("Cole sua API key do Google AI Studio (Enter para pular e usar só o modo por regras): ")
    if _api_key.strip():
        _client = genai.Client(api_key=_api_key)
        _MODEL_DISPONIVEL = True
        print("IA generativa configurada.")
    else:
        print("Nenhuma chave informada — usando apenas o modo por regras (fallback).")
except Exception as e:
    print("Não foi possível configurar a IA generativa, seguindo só com o modo por regras. Detalhe:", e)


def montar_contexto() -> str:
    r = calcular_resumo()
    return (
        f"Perfil do usuário (perfil_investidor.json): {perfil}\n"
        f"Resumo financeiro calculado a partir de transacoes.csv: {r}\n"
        f"Produtos financeiros disponíveis (produtos_financeiros.json): {produtos_data['produtos']}\n"
        f"FAQ: {produtos_data['faq']}"
    )


def responder(pergunta: str) -> str:
    if not pergunta.strip():
        return "Pode repetir sua pergunta? Não recebi nenhum texto."

    if _MODEL_DISPONIVEL:
        try:
            contexto = montar_contexto()
            prompt = f"{SYSTEM_PROMPT}\n\nCONTEXTO:\n{contexto}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA:"
            resposta = _client.models.generate_content(model=_MODEL_NOME, contents=prompt)
            texto = (resposta.text or "").strip()
            if texto:
                return texto
        except Exception:
            pass  # cai no fallback determinístico abaixo

    return responder_regras(pergunta)

Cole sua API key do Google AI Studio (Enter para pular e usar só o modo por regras): ··········
IA generativa configurada.


In [3]:
print("Fin carregado! Teste abaixo")
print(responder("quanto falta pra minha reserva?"))

perguntas_teste = [
    "quanto falta pra minha reserva?",
    "o que é CDI?",
    "posso gastar 900 em viagem pra Ouro Preto?",
    "o que é IVVB11?",
    "me indica 5% ao mês garantido",
    "Juros",
]
for p in perguntas_teste:
    print("\nVocê:", p)
    print("Fin:", responder(p))

Fin carregado! Teste abaixo
Olá Lucas, para atingir sua meta de R$ 10.000,00 da reserva de emergência, ainda faltam R$ 6.800,00. Seu progresso atual é de 32%. 

Fonte: resumo financeiro calculado a partir de transacoes.csv.

Você: quanto falta pra minha reserva?
Fin: Olá, Lucas! Faltam 6.800 para você atingir sua meta de 10.000 para a reserva de emergência.

Fonte: transacoes.csv

Você: o que é CDI?
Fin: O CDI é a taxa de juros praticada entre bancos, funcionando como um referencial para investimentos de renda fixa e sendo muito próxima da Selic. 

Fonte: FAQ.

Você: posso gastar 900 em viagem pra Ouro Preto?
Fin: Olá, Lucas! Com sua renda mensal de R$ 8.500 e a necessidade de atingir sua meta de R$ 10.000 para a reserva de emergência, você deve avaliar se esse gasto não comprometerá o progresso dos seus 32% atuais. Lembre-se de que o foco é atingir os R$ 6.800 que ainda faltam para sua meta.

Fonte: perfil_investidor.json e transacoes.csv

Você: o que é IVVB11?
Fin: O IVVB11 é um ETF 

In [9]:
def chat():
    r = calcular_resumo()
    # Usa .get() com valor padrão pra nunca dar KeyError
    meta = r.get('meta', 10000)
    atual = r.get('atual', 3200)
    falta = r.get('falta', meta - atual)
    progresso = r.get('progresso', atual/meta*100 if meta else 0)
    custo_medio = r.get('custo_medio', r.get('custo', 1048))
    renda = r.get('renda', 8500)
    nome = perfil.get('nome', 'Lucas')

    print(f"Oi {nome}! Eu sou o Fin, seu educador financeiro.")
    print(f"Vi que voce guardou R$ {atual:.0f} de R$ {meta:.0f} pra reserva ({progresso:.0f}%). Faltam R$ {falta:.0f} - ta indo bem!")
    print(f"Seu perfil moderado (score 65) = 70% seguro (Tesouro Selic, CDB) e 30% risco (IVVB11).")
    print(f"Seu custo medio e R$ {custo_medio:.0f}/mes e sua renda total R$ {renda:.0f}.\n")
    print("O que quer fazer hoje? Exemplos:")
    print("- qual melhor investimento hoje?")
    print("- quanto falta pra reserva?")
    print("- quanto posso gastar com lazer?")
    print("- quanto devo gastar com educacao?")
    print("- o que e CDI?")
    print("\nDigite 'sair' para encerrar\n")

    try:
        while True:
            pergunta = input("Você: ")
            if pergunta.lower().strip() in ["sair", "exit", "quit", "tchau", "obrigado"]:
                print(f"\nFin: Valeu {nome}! Faltam R$ {falta:.0f} pra meta. Continue firme! 🚀")
                break
            if not pergunta.strip():
                continue
            print(f"Fin: {responder(pergunta)}\n")
    except KeyboardInterrupt:
        print("\nFin: Chat encerrado.")

chat()

Oi Lucas! Eu sou o Fin, seu educador financeiro.
Vi que voce guardou R$ 3200 de R$ 10000 pra reserva (32%). Faltam R$ 6800 - ta indo bem!
Seu perfil moderado (score 65) = 70% seguro (Tesouro Selic, CDB) e 30% risco (IVVB11).
Seu custo medio e R$ 1225/mes e sua renda total R$ 8500.

O que quer fazer hoje? Exemplos:
- qual melhor investimento hoje?
- quanto falta pra reserva?
- quanto posso gastar com lazer?
- quanto devo gastar com educacao?
- o que e CDI?

Digite 'sair' para encerrar

Fin: O CDI é a taxa de referência usada pelos bancos, enquanto o CDB é um produto de investimento que rende uma porcentagem dessa taxa, como o CDB 110% do CDI disponível. Lembre-se que promessas de rentabilidade garantida fora do contexto são sinal de golpe. 

Fonte: FAQ e produtos_financeiros.json

Fin: Não tenho essa informação. Sei responder sobre reserva de emergência, CDI/Selic, CDB, IVVB11 e orçamento.

Fonte: perfil_investidor.json, transacoes.csv, produtos_financeiros.json e FAQ.

Fin: Não tenho e